# XFeat: 고정 분위수 + packed Thermometer 기준선 (2026-09-30)

이 노트북은 **학습 없이** Frozen XFeat 64차원 descriptor를 고정된 차원별 8구간 분위수로 압축하고, 같은 ABS 거리를 packed Thermometer로 계산합니다. 원본 float cosine, uint8 ABS, packed Thermometer를 동일한 KP와 이미지 쌍에서 비교합니다.

## 지금까지 확인된 성과와 선택

아래는 사용자가 이전 노트북에서 실행한 **2026-09-30 결과 기록**입니다. 새 노트북의 실행 결과가 아닙니다.

| 방법 | 합성 precision | HPatches precision |
|---|---:|---:|
| 고정 분위수 + ABS / Thermometer | 66.93% | 65.90% |
| 학습 경계 + ABS / Thermometer | 66.90% | 65.77% |
| Projection + 고정 분위수 + ABS / Thermometer | 67.87% | 66.02% |

HPatches 고정 설정의 첫 800×800 KP 쌍: ABS 거리 4.457 ms → Thermometer 거리 1.289 ms (**3.46×**), 인코딩 포함 매칭 7.026 → 4.107 ms (**1.71×**). Float cosine + MNN은 1.654 ms. 모든 ABS/Thermometer 거리행렬과 MNN 인덱스가 일치했습니다. 속도는 Numba 2 threads, warmup 3, repeats 12의 중앙값이며 XFeat 추출과 projection은 제외했습니다.

**선택:** 고정 분위수 + Thermometer를 개발 기준선으로 사용하고, float와 ABS를 검증 기준으로 유지합니다. Projection·경계 학습은 실제 HPatches 개선이 작아서 이 노트북의 핵심 경로에서 제외했습니다. 원래 노트북은 그대로 남아 있습니다.

## 실행 방법

위에서 아래로 모두 실행합니다. COCO/HPatches ZIP은 기존 Drive 캐시를 재사용합니다. COCO의 별도 장면에서 경계만 보정하고, 합성 검증 장면 및 HPatches는 경계 계산에 사용하지 않습니다. 24개 COCO 이미지와 기존 합성 warp 설정을 유지합니다. 결과와 경계 파일은 Drive에 실행별 폴더로 저장합니다.

Thermometer: 구간 2=`1100000`, 구간 5=`1111100`, XOR popcount=3=|2−5|. 64차원을 경계별로 묶으면 7개의 uint64 = **56 B/KP**. Float32는 256 B, uint8 bin indices는 64 B. 전체 파이프라인 보관 메모리에는 검증용 float/ABS 사본도 포함되므로 56 B만 사용한다고 해석하지 않습니다.


In [ ]:
!pip -q install opencv-python-headless numba

import time
import io
import zipfile
import subprocess
import shutil
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from numba import njit, prange, set_num_threads, get_num_threads
from google.colab import files, drive

SEED = 7
TOP_K = 800
LEVELS = 8  # seven thresholds per descriptor dimension
CALIBRATION_WARPS_PER_IMAGE = 6
VALIDATION_WARPS_PER_IMAGE = 3
MAX_POSITIVES_PER_PAIR = 500
GEOMETRIC_THRESHOLD = 3.0
NUMBA_THREADS = 2
RUN_HPatches = True
TIMING_PAIRS_PER_DATASET = 5  # evenly spaced pairs; compare every method on each pair
WARMUP = 3
REPEATS = 12

set_num_threads(NUMBA_THREADS)
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

xfeat = torch.hub.load('verlab/accelerated_features', 'XFeat', pretrained=True, top_k=TOP_K).to(device).eval()
for parameter in xfeat.parameters():
    parameter.requires_grad_(False)
print('Device:', device)
# COCO is used for calibration and an internal synthetic-warp check.
# No projection or threshold learning. HPatches is reserved for the external real-photo evaluation at the end.
DATA_SOURCE = 'coco'  # change to 'upload' to use your own JPG/PNG files
COCO_SOURCE_COUNT = 24
COCO_URL = 'http://images.cocodataset.org/zips/val2017.zip'
drive.mount('/content/drive')
data_dir = Path('/content/drive/MyDrive/Colab Notebooks/CSE527_datasets')
data_dir.mkdir(parents=True, exist_ok=True)
print('Persistent dataset cache:', data_dir)

def download_archive(url, destination):
    if destination.exists() and zipfile.is_zipfile(destination):
        print('Using persistent Drive cache:', destination)
        return
    # Reuse a download from an earlier run in this same Colab session, if present.
    legacy = Path('/content/xfeat_data') / destination.name
    if legacy.exists() and zipfile.is_zipfile(legacy):
        shutil.copy2(legacy, destination)
        print('Copied previous runtime download into Drive:', destination)
        return
    print('Downloading once into Drive:', url)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', str(destination), url], check=True)
    if not zipfile.is_zipfile(destination):
        raise RuntimeError(f'Incomplete or invalid ZIP: {destination}. Delete it and rerun this cell.')

if DATA_SOURCE == 'coco':
    coco_zip = data_dir / 'val2017.zip'
    download_archive(COCO_URL, coco_zip)
    with zipfile.ZipFile(coco_zip) as archive:
        names = sorted(n for n in archive.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
        if len(names) < COCO_SOURCE_COUNT:
            raise RuntimeError('COCO archive has fewer images than requested.')
        selected = np.random.default_rng(SEED).choice(len(names), COCO_SOURCE_COUNT, replace=False)
        image_items = [(Path(names[i]).name, archive.read(names[i])) for i in selected]
    print(f'Selected {len(image_items)} COCO images using seed {SEED}; no COCO annotations needed.')
elif DATA_SOURCE == 'upload':
    print('Upload JPG/PNG images from distinct scenes.')
    uploaded = files.upload()
    image_items = list(uploaded.items())
    if not image_items:
        raise ValueError('Upload at least one image.')
else:
    raise ValueError("DATA_SOURCE must be 'coco' or 'upload'")

def decode_and_resize(raw_bytes, maximum_side=640):
    image = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError('An uploaded file is not a readable JPG/PNG image.')
    height, width = image.shape[:2]
    scale = min(1.0, maximum_side / max(height, width))
    new_width = max(64, (int(width * scale) // 32) * 32)
    new_height = max(64, (int(height * scale) // 32) * 32)
    return cv2.resize(image, (new_width, new_height))

# Split by source image, before any synthetic warps or descriptor calibration.
# Upload images of distinct scenes; near-duplicate photos belong to the same split.
image_items = sorted(image_items)
images = [decode_and_resize(raw) for _, raw in image_items]
if len(images) >= 2:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(images))
    validation_count = max(1, int(np.ceil(0.25 * len(images))))
    validation_ids = set(order[:validation_count].tolist())
    calibration_images = [im for i, im in enumerate(images) if i not in validation_ids]
    validation_images = [im for i, im in enumerate(images) if i in validation_ids]
    print('Calibration files:', [name for i, (name, _) in enumerate(image_items) if i not in validation_ids])
    print('Held-out files:', [name for i, (name, _) in enumerate(image_items) if i in validation_ids])
else:
    calibration_images = validation_images = images
    print('WARNING: One source image only; validation shares scene content with training.')
print('Calibration images:', len(calibration_images), 'Validation images:', len(validation_images))

## 1. 특징 추출과 calibration / 검증 분리

XFeat는 고정합니다. 기존 프로토콜과 같은 calibration 이미지의 합성 warp descriptor에서 분위수 경계를 계산합니다. 대응점 계산은 기존 샘플링 순서를 유지하기 위한 것이며, 가중치를 학습하지 않습니다.


In [ ]:
def image_to_tensor(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()[None].to(device) / 255.0

@torch.inference_mode()
def extract_features(image):
    output = xfeat.detectAndCompute(image_to_tensor(image), top_k=TOP_K)[0]
    return {
        'keypoints': output['keypoints'].detach().cpu().float(),
        'descriptors': F.normalize(output['descriptors'].detach().cpu().float(), dim=-1),
    }

def make_random_homography(height, width, generator):
    center = (width / 2.0, height / 2.0)
    angle = generator.uniform(-20.0, 20.0)
    scale = generator.uniform(0.88, 1.08)
    affine = cv2.getRotationMatrix2D(center, angle, scale)
    affine[0, 2] += generator.uniform(-0.06, 0.06) * width
    affine[1, 2] += generator.uniform(-0.06, 0.06) * height
    homography = np.vstack([affine, [0., 0., 1.]]).astype(np.float32)
    homography[2, 0] = generator.uniform(-7e-5, 7e-5)
    homography[2, 1] = generator.uniform(-7e-5, 7e-5)
    return homography

def warp_image(image, homography, generator):
    height, width = image.shape[:2]
    warped = cv2.warpPerspective(image, homography, (width, height),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    alpha = float(generator.uniform(0.82, 1.12))
    beta = float(generator.uniform(-12., 18.))
    return np.clip(warped.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

def project_points(points, homography):
    p = points.detach().cpu().numpy()
    p_h = np.concatenate([p, np.ones((len(p), 1), np.float32)], axis=1)
    projected = (homography @ p_h.T).T
    return torch.from_numpy((projected[:, :2] / projected[:, 2:3]).astype(np.float32))

def ground_truth_pairs(keypoints_1, keypoints_2, homography):
    if not len(keypoints_1) or not len(keypoints_2):
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    distances_12 = torch.cdist(project_points(keypoints_1, homography), keypoints_2)
    nearest_12 = distances_12.argmin(dim=1)
    errors = distances_12[torch.arange(len(keypoints_1)), nearest_12]
    distances_21 = torch.cdist(project_points(keypoints_2, np.linalg.inv(homography)), keypoints_1)
    nearest_21 = distances_21.argmin(dim=1)
    indices = torch.arange(len(keypoints_1))
    good = (nearest_21[nearest_12] == indices) & (errors < GEOMETRIC_THRESHOLD)
    return indices[good], nearest_12[good]

def build_pairs(source_images, warps_per_image, generator):
    pairs = []
    for image_index, image in enumerate(source_images):
        base = extract_features(image)
        for warp_index in range(warps_per_image):
            h = make_random_homography(*image.shape[:2], generator)
            warped = extract_features(warp_image(image, h, generator))
            i, j = ground_truth_pairs(base['keypoints'], warped['keypoints'], h)
            if len(i) > MAX_POSITIVES_PER_PAIR:
                keep = torch.randperm(len(i))[:MAX_POSITIVES_PER_PAIR]
                i, j = i[keep], j[keep]
            if len(i) < 32:
                print(f'Warning: image {image_index}, warp {warp_index}: {len(i)} positive pairs')
            pairs.append({'d1': base['descriptors'], 'd2': warped['descriptors'],
                          'k1': base['keypoints'], 'k2': warped['keypoints'],
                          'pos1': i, 'pos2': j, 'H': h})
    return pairs

start = time.perf_counter()
calibration_pairs = build_pairs(calibration_images, CALIBRATION_WARPS_PER_IMAGE, rng)
validation_pairs = build_pairs(validation_images, VALIDATION_WARPS_PER_IMAGE, rng)
calibration = torch.cat([torch.cat([p['d1'], p['d2']], dim=0) for p in calibration_pairs], dim=0)
if len(calibration) > 50000:
    calibration = calibration[torch.randperm(len(calibration))[:50000]]
print('Feature extraction time (calibration and validation):', round(time.perf_counter() - start, 2), 's')
print('Calibration pairs:', len(calibration_pairs), 'Validation pairs:', len(validation_pairs),
      'Calibration descriptors:', len(calibration))

In [ ]:
# Fixed training-partition quantile edges; no optimizer, projection, or threshold learning.
base_edges_cpu = torch.quantile(calibration, torch.arange(1, LEVELS).float() / LEVELS, dim=0).T.contiguous()

def hard_quantize(descriptors, edges):
    return (descriptors[:, :, None] > edges[None, :, :]).sum(dim=-1).to(torch.uint8)

print('Fixed edges:', tuple(base_edges_cpu.shape), '| learned parameters: 0')


## 2. 검증 기준 커널과 packed Thermometer

ABS와 Thermometer는 같은 8구간 L1 거리입니다. 모든 거리행렬과 MNN 인덱스를 검사합니다. popcount는 LLVM ctpop을 사용합니다. 이 셀에서는 pruning을 하지 않습니다.


In [ ]:
@njit(parallel=True, cache=False)
def pairwise_uint8_abs(code_1, code_2):
    out = np.empty((code_1.shape[0], code_2.shape[0]), np.uint16)
    for i in prange(code_1.shape[0]):
        for j in range(code_2.shape[0]):
            total = 0
            for k in range(code_1.shape[1]):
                a, b = int(code_1[i, k]), int(code_2[j, k])
                total += a - b if a >= b else b - a
            out[i, j] = total
    return out

def mutual_from_scores(scores, higher_is_better):
    if not scores.shape[0] or not scores.shape[1]:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
    if higher_is_better:
        nearest_12, nearest_21 = scores.argmax(axis=1), scores.argmax(axis=0)
    else:
        nearest_12, nearest_21 = scores.argmin(axis=1), scores.argmin(axis=0)
    indices = np.arange(scores.shape[0])
    keep = nearest_21[nearest_12] == indices
    return indices[keep], nearest_12[keep]

def encode(desc, edges):
    with torch.no_grad():
        levels = hard_quantize(desc.to(device), edges.to(device))
    return np.ascontiguousarray(levels.cpu().numpy(), dtype=np.uint8)

def match_float(pair):
    scores = (pair['d1'] @ pair['d2'].T).numpy()
    return mutual_from_scores(scores, higher_is_better=True)

def match_abs(pair, edges):
    a, b = encode(pair['d1'], edges), encode(pair['d2'], edges)
    return mutual_from_scores(pairwise_uint8_abs(a, b), higher_is_better=False)


# Packed thermometer: same 8-bin L1 distance, different computation.
from numba import types
from numba.extending import intrinsic
from llvmlite import ir

@intrinsic
def thermometer_popcount64(typing_context, value_type):
    if value_type != types.uint64:
        raise TypeError('thermometer_popcount64 expects uint64')
    signature = types.uint64(types.uint64)
    def codegen(context, builder, signature, arguments):
        function = builder.module.declare_intrinsic('llvm.ctpop', [ir.IntType(64)])
        return builder.call(function, arguments)
    return signature, codegen

@njit(cache=False)
def _pack_thermometer64(codes):
    packed = np.zeros((codes.shape[0], 7), dtype=np.uint64)
    for row in range(codes.shape[0]):
        for threshold in range(1, 8):
            word = np.uint64(0)
            for dimension in range(64):
                if codes[row, dimension] >= threshold:
                    word |= np.uint64(1) << np.uint64(dimension)
            packed[row, threshold - 1] = word
    return packed

def pack_thermometer64(codes):
    codes = np.asarray(codes)
    if codes.ndim != 2 or codes.shape[1] != 64:
        raise ValueError('Expected (N, 64) bin indices')
    if not np.issubdtype(codes.dtype, np.integer):
        raise ValueError('Bin indices must be integers')
    if codes.size and (codes.min() < 0 or codes.max() > 7):
        raise ValueError('Expected bin indices 0..7')
    return _pack_thermometer64(np.ascontiguousarray(codes, dtype=np.uint8))

@njit(parallel=True, cache=False)
def pairwise_thermometer64(packed_1, packed_2):
    distances = np.empty((packed_1.shape[0], packed_2.shape[0]), dtype=np.uint16)
    for i in prange(packed_1.shape[0]):
        for j in range(packed_2.shape[0]):
            total = np.uint64(0)
            for plane in range(7):
                total += thermometer_popcount64(packed_1[i, plane] ^ packed_2[j, plane])
            distances[i, j] = total
    return distances

def match_thermometer(pair, edges):
    a = pack_thermometer64(encode(pair['d1'], edges))
    b = pack_thermometer64(encode(pair['d2'], edges))
    return mutual_from_scores(pairwise_thermometer64(a, b), higher_is_better=False)

def verify_thermometer_codes(a, b):
    abs_distances = pairwise_uint8_abs(a, b)
    thermo_distances = pairwise_thermometer64(pack_thermometer64(a), pack_thermometer64(b))
    np.testing.assert_array_equal(abs_distances, thermo_distances)
    if len(a) and len(b):
        abs_matches = mutual_from_scores(abs_distances, False)
        thermo_matches = mutual_from_scores(thermo_distances, False)
        for abs_indices, thermo_indices in zip(abs_matches, thermo_matches):
            np.testing.assert_array_equal(abs_indices, thermo_indices)
    return True

if LEVELS != 8:
    raise ValueError('This experiment is for 8 bins and 64 dimensions')
# Compile before measurements; test all 8x8 scalar values at bits 0 and 63.
for dimension in (0, 63):
    scalar_codes = np.zeros((8, 64), dtype=np.uint8)
    scalar_codes[:, dimension] = np.arange(8, dtype=np.uint8)
    verify_thermometer_codes(scalar_codes, scalar_codes)
    expected = np.abs(np.arange(8)[:, None] - np.arange(8)[None, :])
    np.testing.assert_array_equal(pairwise_thermometer64(
        pack_thermometer64(scalar_codes), pack_thermometer64(scalar_codes)), expected)
test_rng = np.random.default_rng(SEED)
verify_thermometer_codes(test_rng.integers(0, 8, (19, 64), dtype=np.uint8),
                         test_rng.integers(0, 8, (23, 64), dtype=np.uint8))
zeros = np.zeros((1, 64), dtype=np.uint8)
sevens = np.full((1, 64), 7, dtype=np.uint8)
verify_thermometer_codes(zeros, sevens)
assert int(pairwise_thermometer64(pack_thermometer64(zeros), pack_thermometer64(sevens))[0, 0]) == 448
assert pack_thermometer64(zeros).nbytes == 56
print('PASS: scalar cases, highest bit, random distances/MNN, maximum distance 448.')
print('Storage per descriptor: float32 256 B | uint8 ABS 64 B | packed thermometer 56 B.')
print('Thermometer is 7 threshold planes, not the 24 B packed 3-bit numeric representation.')


## 3. 외부 HPatches 쌍 구성

6개 viewpoint + 6개 illumination sequence에서 각각 1→2, 1→3: 총 24쌍. 리사이즈에 맞게 homography를 보정합니다. 경계 calibration에는 사용하지 않습니다. 이 평가는 리사이즈 좌표에서 3px 기준의 이미지 쌍 평가이며 공식 HPatches patch benchmark 점수는 아닙니다.


In [ ]:
real_pairs = []
if RUN_HPatches:
    HPATCHES_URL = 'https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'
    HPATCHES_SEQUENCES_PER_TYPE = 6  # 6 viewpoint + 6 illumination scenes
    HPATCHES_TARGETS = (2, 3)       # two real photos per sequence
    hpatches_zip = data_dir / 'hpatches-sequences-release.zip'
    download_archive(HPATCHES_URL, hpatches_zip)

    def decode_hpatches_with_scale(raw_bytes):
        original = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
        if original is None:
            raise RuntimeError('Failed to decode HPatches PPM image')
        old_h, old_w = original.shape[:2]
        resized = decode_and_resize(raw_bytes)
        new_h, new_w = resized.shape[:2]
        scale = np.diag([new_w / old_w, new_h / old_h, 1.0]).astype(np.float64)
        return resized, scale

    real_pairs = []
    with zipfile.ZipFile(hpatches_zip) as archive:
        entries = set(archive.namelist())
        folders = sorted({name.rsplit('/', 1)[0] for name in entries if name.endswith('/1.ppm')})
        for prefix in ('v_', 'i_'):
            selected_folders = [folder for folder in folders if folder.split('/')[-1].startswith(prefix)]
            if len(selected_folders) < HPATCHES_SEQUENCES_PER_TYPE:
                raise RuntimeError(f'Expected more HPatches {prefix} folders; found {len(selected_folders)}')
            for folder in selected_folders[:HPATCHES_SEQUENCES_PER_TYPE]:
                image_1, scale_1 = decode_hpatches_with_scale(archive.read(folder + '/1.ppm'))
                features_1 = extract_features(image_1)
                for target_id in HPATCHES_TARGETS:
                    image_name = f'{folder}/{target_id}.ppm'
                    h_name = f'{folder}/H_1_{target_id}'
                    if image_name not in entries or h_name not in entries:
                        raise RuntimeError(f'Missing HPatches image or homography: {folder}, {target_id}')
                    image_2, scale_2 = decode_hpatches_with_scale(archive.read(image_name))
                    features_2 = extract_features(image_2)
                    original_h = np.loadtxt(io.BytesIO(archive.read(h_name))).reshape(3, 3)
                    resized_h = scale_2 @ original_h @ np.linalg.inv(scale_1)
                    real_pairs.append({
                        'name': f'{folder.split("/")[-1]}:1->{target_id}',
                        'group': prefix, 'd1': features_1['descriptors'],
                        'd2': features_2['descriptors'],
                        'k1': features_1['keypoints'], 'k2': features_2['keypoints'],
                        'H': resized_h,
                    })
    print('HPatches real-photo pairs:', len(real_pairs), '(viewpoint and illumination)')



## 4. 코드 재사용과 다음 개선용 인터페이스

이미지의 float descriptor를 양자화하고 packing한 결과를 메모리에 보관해 여러 matcher가 재사용합니다. 재실행 시에는 다시 특징을 추출하며, ZIP만 영구 캐시합니다.

`MATCHERS`의 함수는 prepared pair를 입력받아 `(i, j)` MNN 인덱스를 반환합니다. 아래 EXACT_COMPRESSED_METHODS에 새 이름을 추가하면 uint8 ABS와의 일치 여부가 평가됩니다. 다음 구현도 동일한 argmin 동점 규칙(가장 작은 인덱스)을 유지해야 합니다.


In [ ]:
def prepare_pair(pair):
    a, b = encode(pair['d1'], base_edges_cpu), encode(pair['d2'], base_edges_cpu)
    return {**pair, 'a': a, 'b': b,
            'packed_a': pack_thermometer64(a), 'packed_b': pack_thermometer64(b)}

prepared_datasets = {'synthetic': [prepare_pair(p) for p in validation_pairs]}
if real_pairs:
    prepared_datasets['HPatches'] = [prepare_pair(p) for p in real_pairs]

MATCHERS = {
    'float cosine': lambda p: match_float(p),
    'uint8 ABS': lambda p: mutual_from_scores(pairwise_uint8_abs(p['a'], p['b']), False),
    'packed thermometer': lambda p: mutual_from_scores(
        pairwise_thermometer64(p['packed_a'], p['packed_b']), False),
}
EXACT_COMPRESSED_METHODS = {'uint8 ABS', 'packed thermometer'}
# Next: define fused_mnn(prepared_pair), then register it here and in the exact set.
# MATCHERS['fused thermometer MNN'] = fused_mnn
# EXACT_COMPRESSED_METHODS.add('fused thermometer MNN')

def checked_compressed_matches(name, prepared_pair):
    reference = MATCHERS['uint8 ABS'](prepared_pair)
    candidate = MATCHERS[name](prepared_pair)
    for expected, actual in zip(reference, candidate):
        np.testing.assert_array_equal(expected, actual, err_msg=name)
    return candidate

print('Prepared datasets:', {k: len(v) for k, v in prepared_datasets.items()})
print('Matcher registry:', list(MATCHERS))


In [ ]:
def evaluate_dataset(label, pairs):
    aggregate = {name: {'mutual': 0, 'correct': 0, 'errors': [], 'per_pair': []}
                 for name in MATCHERS}
    for pair_number, pair in enumerate(pairs):
        np.testing.assert_array_equal(pairwise_uint8_abs(pair['a'], pair['b']),
                                      pairwise_thermometer64(pair['packed_a'], pair['packed_b']))
        for name, matcher in MATCHERS.items():
            i, j = (checked_compressed_matches(name, pair)
                    if name in EXACT_COMPRESSED_METHODS else matcher(pair))
            errors = np.linalg.norm(project_points(pair['k1'][i], pair['H']).numpy()
                                    - pair['k2'][j].numpy(), axis=1)
            correct = int(np.count_nonzero(errors < GEOMETRIC_THRESHOLD))
            item = aggregate[name]
            item['mutual'] += len(i); item['correct'] += correct
            item['errors'].extend(errors.tolist())
            item['per_pair'].append({'name': pair.get('name', str(pair_number)),
                                     'group': pair.get('group', 'synthetic'),
                                     'mutual': len(i), 'correct': correct})
    records = []
    for name, item in aggregate.items():
        errors = item.pop('errors')
        item.update(dataset=label, method=name,
                    precision=100*item['correct']/item['mutual'] if item['mutual'] else None,
                    median_px=float(np.median(errors)) if errors else None,
                    exact_abs_mnn_verified=name in EXACT_COMPRESSED_METHODS)
        records.append(item)
        print(label, '|', name, '|', item['correct'], '/', item['mutual'],
              '| precision:', item['precision'], '| median px:', item['median_px'])
    print('PASS: ABS/Thermometer full-distance and exact-method MNN checks on every pair.')
    return records

accuracy_results = [record for label, pairs in prepared_datasets.items()
                    for record in evaluate_dataset(label, pairs)]


## 5. 속도: 미리 압축한 코드 / 압축을 포함한 전체 매칭

데이터셋당 최대 5개 쌍을 균등 간격으로 선택하고, 각 쌍에서 동일하게 warmup 3 / repeats 12 중앙값을 측정합니다. XFeat 추출은 제외합니다. 코드 재사용 매칭과 매번 인코딩하는 매칭을 구분합니다. Float는 원본 64차원 cosine 기준선입니다. 개별 측정의 중앙값들을 더해서 전체 시간을 추정하지 않습니다.


In [ ]:
def benchmark(fn, warmup=WARMUP, repeats=REPEATS):
    for _ in range(warmup): fn()
    samples = []
    for _ in range(repeats):
        start = time.perf_counter_ns(); fn()
        samples.append((time.perf_counter_ns() - start) / 1e6)
    return float(np.median(samples))

def compress_both(pair):
    return (pack_thermometer64(encode(pair['d1'], base_edges_cpu)),
            pack_thermometer64(encode(pair['d2'], base_edges_cpu)))

timing_results = []
for label, pairs in prepared_datasets.items():
    indices = np.unique(np.linspace(0, len(pairs)-1,
                        min(TIMING_PAIRS_PER_DATASET, len(pairs)), dtype=int)) if pairs else []
    for index in indices:
        p = pairs[index]
        measurements = {
            'abs_distance_only_ms': benchmark(lambda: pairwise_uint8_abs(p['a'], p['b'])),
            'thermometer_distance_only_ms': benchmark(lambda: pairwise_thermometer64(p['packed_a'], p['packed_b'])),
            'bin_encoding_both_sets_ms': benchmark(lambda: (encode(p['d1'], base_edges_cpu), encode(p['d2'], base_edges_cpu))),
            'packing_both_sets_ms': benchmark(lambda: (pack_thermometer64(p['a']), pack_thermometer64(p['b']))),
            'thermometer_compression_both_sets_ms': benchmark(lambda: compress_both(p)),
            'abs_encode_and_mnn_ms': benchmark(lambda: match_abs(p, base_edges_cpu)),
            'thermometer_encode_pack_and_mnn_ms': benchmark(lambda: match_thermometer(p, base_edges_cpu)),
        }
        for name, matcher in MATCHERS.items():
            measurements[name + ' cached_mnn_ms'] = benchmark(lambda matcher=matcher: matcher(p))
        timing_results.append({'dataset': label, 'pair_index': int(index),
                               'name': p.get('name', str(index)), 'n': len(p['a']), 'm': len(p['b']),
                               **measurements})
        print(label, 'pair', index, '|', len(p['a']), 'x', len(p['b']))
        for name, value in measurements.items(): print(f'  {name}: {value:.3f} ms')
    subset = [r for r in timing_results if r['dataset'] == label]
    if subset:
        print(label, 'summary: median of per-pair medians, min..max (ms)')
        for key in measurements:
            values = [r[key] for r in subset]
            print(f'  {key}: {np.median(values):.3f} ({min(values):.3f}..{max(values):.3f})')


## 6. 다음 개선 순서와 합격 기준

1. **Thermometer 거리 + 양방향 MNN 통합**: 거리행렬을 저장하지 않고 최선 후보를 갱신. 모든 이미지 쌍에서 기존 ABS의 MNN 인덱스와 동일해야 합니다. 병렬 갱신 충돌을 피하고 동점 규칙을 유지합니다.
2. **부분합 pruning**: 아직 계산하지 않은 plane 기여는 비음수. 현재 부분합이 양쪽 최선 거리보다 모두 클 때 중단할 수 있습니다. 즉 양방향 top-1에서는 `partial > max(best_A[i], best_B[j])`. pruning은 압축 ABS 거리 기준으로만 보장됩니다. top-2 ratio를 추가하면 별도 기준이 필요합니다.
3. **비교 항 수 기록**: 7개 plane 중 실제로 계산한 개수와 중단 후보 비율을 정확도 검증과 함께 기록합니다. 계측 커널과 순수 타이밍 커널은 분리합니다.
4. 정확성을 보존하는 개선을 평가한 뒤 **64비트 coarse → top-K refinement**를 별도 근사 실험으로 추가합니다. 이때 candidate recall@K와 매칭 손실을 반드시 기록합니다.

합격 기준: 거리/MNN 검증 통과, 모든 방법에 같은 KP와 경계 적용, 여러 쌍에서 cached 및 encoding-inclusive 시간이 실제로 감소. 분기와 후보 갱신 비용 때문에 pruning이 오히려 느릴 수 있으므로 측정으로 판단합니다. 지금 파일에는 위 개선을 아직 구현하지 않았습니다.


In [ ]:
import json, platform
from datetime import datetime
from zoneinfo import ZoneInfo
run_id = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
result_dir = data_dir / 'experiment_results' / ('fixed_thermometer_' + run_id)
result_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = result_dir / 'fixed_quantile_edges.pt'
torch.save({'edges': base_edges_cpu, 'levels': LEVELS, 'dimensions': 64,
            'seed': SEED, 'note': 'Calibration scenes only, no learned projection'}, checkpoint_path)
report = {
    'experiment': 'XFeat fixed quantile packed thermometer baseline', 'run_id_seoul': run_id,
    'seed': SEED, 'top_k': TOP_K, 'levels': LEVELS, 'dimensions': 64,
    'coco_source_count': len(images), 'calibration_warps_per_image': CALIBRATION_WARPS_PER_IMAGE,
    'validation_warps_per_image': VALIDATION_WARPS_PER_IMAGE,
    'source_names': [name for name, _ in image_items],
    'calibration_source_names': [name for i, (name, _) in enumerate(image_items) if i not in validation_ids] if len(images)>=2 else [image_items[0][0]],
    'validation_source_names': [name for i, (name, _) in enumerate(image_items) if i in validation_ids] if len(images)>=2 else [image_items[0][0]],
    'geometric_threshold_resized_px': GEOMETRIC_THRESHOLD,
    'numba_threads': get_num_threads(), 'torch_threads': torch.get_num_threads(),
    'device_for_extraction_and_encoding': device, 'platform': platform.platform(),
    'versions': {'numpy': np.__version__, 'torch': torch.__version__},
    'warmup': WARMUP, 'repeats': REPEATS,
    'bytes_per_descriptor': {'float32': 256, 'uint8_abs': 64, 'packed_thermometer': 56},
    'storage_note': 'Payload only. Notebook retains all representations for verification.',
    'projection': False, 'threshold_learning': False, 'pruning': False,
    'timing_note': 'XFeat excluded; cached MNN and encoding-inclusive MNN measured separately.',
    'accuracy_results': accuracy_results, 'timing_results': timing_results,
}
result_path = result_dir / 'results.json'
with open(result_path, 'w') as out: json.dump(report, out, indent=2, allow_nan=False)
print('Saved results:', result_path)
print('Saved fixed edges:', checkpoint_path)
